# Indexes
Indexes are often used as parts of price formulas to define the price of energy. They can be based on various data sources, such as historical prices, market data, or even custom calculations. The `Index` class serves as a base class for different types of indexes, providing a common interface for retrieving values based on timestamps and resolutions.

If you want to use an index in a priceformula, it has to be registered by name first. This way the formula can fetch the index values when needed.

A simple index can be provided as a CSV or YAML file, which can be loaded using the `CSVIndex` or `YAMLIndex` classes.

In [1]:
from energy_cost.index import CSVIndex, Index, YAMLIndex

Index.register("foo_csv_index", CSVIndex("../data/indexes/foo.csv"))
Index.register("foo_yaml_index", YAMLIndex("../data/indexes/foo.yml"))

After registering the indexes, you can use them in price formulas by referencing their names. For example, if you have registered an index named "foo_csv_index", you can use it in a price formula like this:


In [2]:
import datetime as dt

from energy_cost.price_formula import IndexAdder, PriceFormula

price_formula = PriceFormula(
    constant_cost=10.0,
    variable_costs=[IndexAdder(index="foo_csv_index", scalar=0.45), IndexAdder(index="foo_yaml_index", scalar=0.55)],
)

price_formula.get_values(
    start=dt.datetime(2025, 12, 25, tzinfo=dt.UTC),
    end=dt.datetime(2026, 1, 5, tzinfo=dt.UTC),
    resolution=dt.timedelta(hours=1),
)

,timestamp,value
0,2025-12-25 00:00:00+00:00,110.0
1,2025-12-25 01:00:00+00:00,110.0
2,2025-12-25 02:00:00+00:00,110.0
3,2025-12-25 03:00:00+00:00,110.0
4,2025-12-25 04:00:00+00:00,110.0
...,...,...
259,2026-01-04 19:00:00+00:00,130.0
260,2026-01-04 20:00:00+00:00,130.0
261,2026-01-04 21:00:00+00:00,130.0
262,2026-01-04 22:00:00+00:00,130.0


We also have a built-in index for the day-ahead prices from ENTSO-E, which can be accessed using the `EntsoeDayAheadIndex` class. This index fetches the day-ahead prices for a specified region.

For more advanced case you can use our `DataFrameIndex` class to create custom indexes based on any DataFrame that contains a "timestamp" and "value" column. Or you can create your own custom index by subclassing the `Index` class and implementing the `get_values` method to fetch the data from your desired source.

In [3]:
from os import environ

import pandas as pd

from energy_cost.index import EntsoeDayAheadIndex
from energy_cost.index.dataframe_index import DataFrameIndex

Index.register("Belpex15min", EntsoeDayAheadIndex(country_code="BE", api_key=environ["ENTSOE_API_KEY"]))

df = pd.DataFrame(
    {
        "timestamp": pd.date_range(start="2025-12-25", end="2026-01-05", freq="15min", tz="UTC"),
        "value": range(0, 24 * 4 * 11 + 1, 1),  # 15-minute intervals for 11 days
    }
)

Index.register("custom_df_index", DataFrameIndex(df))